In [1]:
import numpy as np
import os
from skimage import io
from skimage.transform import resize


In [2]:
input_landslide_images='./Bijie-landslide-dataset/landslide/image/'
input_landslide_dem='./Bijie-landslide-dataset/landslide/dem/'
input_landslide_dem_resized='./Bijie-landslide-dataset/landslide/dem_resized/'
input_landslide_mask='./Bijie-landslide-dataset/landslide/mask/'
input_landslide_images_crop='./Bijie-landslide-dataset/landslide/image_crop/'
input_landslide_dem_resized_crop='./Bijie-landslide-dataset/landslide/dem_resized_crop/'
input_landslide_mask_crop='./Bijie-landslide-dataset/landslide/mask_crop/'


input_nolandslide_images='./Bijie-landslide-dataset/non-landslide/image/'
input_nolandslide_dem='./Bijie-landslide-dataset/non-landslide/dem/'
input_nolandslide_dem_resized='./Bijie-landslide-dataset/non-landslide/dem_resized'
input_nolandslide_images_crop='./Bijie-landslide-dataset/non-landslide/image_crop/'
input_nolandslide_dem_resized_crop='./Bijie-landslide-dataset/non-landslide/dem_resized_crop'


# Images sizes and resize
### resize DEM to initial images

In [3]:
def resize_dem(input_dem,input_image,output_folder):
    """
    Resize dem images to initial images and save to output_folder
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print('creation of ',output_folder)
    names=os.listdir(input_dem)
    
    for name in names:
        s_ima=io.imread('%s/%s'%(input_image,name)).shape
        dem=io.imread('%s/%s'%(input_dem,name))
        dem_resized = resize(dem, s_ima,anti_aliasing=True)
        name_save='%s/%s'%(output_folder,name)
        data=np.clip(dem_resized*255,0,255).astype(np.uint8)
        io.imsave(name_save,data)
    print('converted %d files'%len(names))

In [4]:
resize_dem(input_landslide_dem,input_landslide_images,input_landslide_dem_resized)

converted 770 files


In [6]:
%%capture --no-display
resize_dem(input_nolandslide_dem,input_nolandslide_images,input_nolandslide_dem_resized)

### check consistency with image tiles and get min,max sizes

In [7]:
def check_size_consistency1(input_images,input_dsm,input_mask):
    names=os.listdir(input_dsm)
    max_lig = 0
    max_col = 0
    min_lig=9999999
    min_col=9999999
    for name in names:
        s_ima=io.imread('%s/%s'%(input_images,name)).shape
        s_dsm=io.imread('%s/%s'%(input_dsm,name)).shape
        s_mas=io.imread('%s/%s'%(input_mask,name)).shape
        condition = ((s_ima[1]==s_dsm[1]) and (s_ima[0]==s_dsm[0])and  (s_ima[1]==s_mas[1]) and (s_ima[0]==s_mas[0]))
        if condition is False:
            print('err size for',name)
            print('size mask : ',s_mas)
            print('size dsm : ',s_dsm)
            print('size image : ',s_ima)
        if s_ima[0] > max_lig:
            max_lig=s_ima[0]
        if s_ima[1] > max_col:
            max_col=s_ima[1]
        if s_ima[0] < min_lig:
            min_lig=s_ima[0]
        if s_ima[1] < min_col:
            min_col=s_ima[1]
       
            
    print('checked ',len(names),' images')
    return min_lig,min_col,max_lig,max_col
        
def check_size_consistency2(input_images,input_dsm):
    names=os.listdir(input_dsm)
    max_lig = 0
    max_col = 0
    min_lig=9999999
    min_col=9999999
    for name in names:
        s_ima=io.imread('%s/%s'%(input_images,name)).shape
        s_dsm=io.imread('%s/%s'%(input_dsm,name)).shape
        condition = ((s_ima[1]==s_dsm[1]) and (s_ima[0]==s_dsm[0]))
        if condition is False:
            print('err size for',name)
            print('size dsm : ',s_dsm)
            print('size image : ',s_ima)
        if s_ima[0] > max_lig:
            max_lig=s_ima[0]
        if s_ima[1] > max_col:
            max_col=s_ima[1]
        if s_ima[0] < min_lig:
            min_lig=s_ima[0]
        if s_ima[1] < min_col:
            min_col=s_ima[1]

    print('checked ',len(names),' images')
    return min_lig,min_col,max_lig,max_col


In [8]:
min_lig,min_col,max_lig,max_col=check_size_consistency1(input_landslide_images,input_landslide_dem_resized,input_landslide_mask)

checked  770  images


In [9]:
min_lig,min_col,max_lig,max_col=check_size_consistency2(input_nolandslide_images,input_nolandslide_dem_resized)

checked  2003  images


### crop large images

In [10]:
def sliding_window(image, stepSize, windowSize):
  for y in range(0, image.shape[0], stepSize):
    for x in range(0, image.shape[1], stepSize):
      yield (x, y, image[y:y + windowSize[1], x:x + windowSize[0]])

In [11]:
def crop_image(name_im,dest_name,windowsize,overlap):
    """
    crop an image (png or jpg) with overlap and a given sliding window
    save results in dest_name

    """
    if not os.path.exists(dest_name):
        os.makedirs(dest_name)
        print('creation of ',dest_name)
    im=io.imread(name_im)
    windows=sliding_window(im, windowsize-overlap, (windowsize,windowsize))
    
    # generic image name and extension
    name=os.path.splitext(os.path.split(name_im)[1])[0]
    ext=os.path.splitext(os.path.split(name_im)[1])[1]
    num_im=0
    for window in windows:
        name_save='%s/%s_%.4d%s'%(dest_name,name,num_im,ext)
        io.imsave(name_save,window[2])
        num_im+=1


test

In [12]:
name_im='%s/ny050.png'%input_landslide_dem_resized
os.path.splitext(os.path.split(name_im)[1])[1]
print(name_im)
crop_image(name_im,'/Users/corpetti/tmpo/zfdsdf_dsm/',256,0)

./Bijie-landslide-dataset/landslide/dem_resized//ny050.png


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_16136/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0004.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_16136/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0012.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_16136/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0016.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_16136/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0017.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_16136/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0018.png is a low contrast image
  io.imsave(name_save,window

### crop all small images in a given folder

In [13]:
def crop_images(input_folder,output_folder,window_size,overlap):
    """
    put all images in a given folder
    """
    names=os.listdir(input_folder)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print('creation of ',output_folder)

    for name in names:
        s_ima=io.imread('%s/%s'%(input_folder,name)).shape
        if (s_ima[0]> window_size) or (s_ima[1]> window_size):
            crop_image('%s/%s'%(input_folder,name),output_folder,window_size,overlap)
            #print('image',name,'cropped (size =',s_ima,')')
        else:
            commande='cp %s/%s %s'%(input_folder,name,output_folder)
            #print('image',name,'copied (size =',s_ima,')')
            os.system(commande)


In [14]:
%%capture --no-display
crop_images(input_landslide_images,input_landslide_images_crop,256,0)

In [15]:
%%capture --no-display
crop_images(input_landslide_dem_resized,input_landslide_dem_resized_crop,256,0)

In [16]:
%%capture --no-display
crop_images(input_landslide_mask,input_landslide_mask_crop,256,0)

In [17]:
%%capture --no-display
crop_images(input_nolandslide_images,input_nolandslide_images_crop,256,0)

In [18]:
%%capture --no-display
crop_images(input_nolandslide_dem_resized,input_nolandslide_dem_resized_crop,256,0)
